## [N] Xử lý vấn đề chất lượng dữ liệu 

* **Loại bỏ đặc trưng định danh:** Loại bỏ cột `customer_id` ra khỏi tập dữ liệu đầu vào $\mathbf{X}$ trước khi huấn luyện để tránh việc mô hình ghi nhớ mã hồ sơ.
* **Chiến lược xử lý ngoại lai & chênh lệch thang đo:** 
  * Giữ nguyên các điểm ngoại lai hợp lệ (vì đây là các khách hàng có tài sản lớn/nợ lớn thực tế trong hoạt động ngân hàng).
  * Sử dụng bộ chuẩn hóa **`StandardScaler`** đóng gói trong Pipeline để đưa các đặc trưng số về cùng phân bố chuẩn $\mathcal{N}(0, 1)$, triệt tiêu tác động của chênh lệch biên độ số học.
* **Chiến lược mã hóa biến phân loại:**
  * Áp dụng **`OneHotEncoder(drop='first', sparse_output=False)`** cho 3 biến phân loại (`occupation_status`, `product_type`, `loan_intent`), tham số `drop='first'` giúp chống hiện tượng đa cộng tuyến hoàn hảo (Dummy Variable Trap).

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

df = pd.read_csv('../data/Loan_approval_data_2025.csv')

X = df.drop(columns=['customer_id', 'loan_status']).copy()
y = df['loan_status'].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("=" * 85)
print(f"KÍCH THƯỚC CÁC TẬP DỮ LIỆU SAU PHÂN CHIA (70% Train - 15% Val - 15% Test):")
print(f" - Tập Huấn luyện (Train) : {X_train.shape[0]:,} mẫu ({X_train.shape[0]/len(df):.1%}) | Tỷ lệ duyệt: {y_train.mean():.2%}")
print(f" - Tập Kiểm định  (Val)   : {X_val.shape[0]:,} mẫu  ({X_val.shape[0]/len(df):.1%}) | Tỷ lệ duyệt: {y_val.mean():.2%}")
print(f" - Tập Kiểm tra   (Test)  : {X_test.shape[0]:,} mẫu  ({X_test.shape[0]/len(df):.1%}) | Tỷ lệ duyệt: {y_test.mean():.2%}")
print("=" * 85)

cat_features = ['occupation_status', 'product_type', 'loan_intent']
num_features = [col for col in X.columns if col not in cat_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

df_train_proc = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
df_val_proc   = pd.DataFrame(X_val_processed, columns=feature_names, index=X_val.index)
df_test_proc  = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("\nKẾT QUẢ TIỀN XỬ LÝ MA TRẬN ĐẶC TRƯNG:")
print(f" - Tổng số đặc trưng sau mã hóa: {len(feature_names)} đặc trưng")
print(f" - Kích thước tập Train sau tiền xử lý: {df_train_proc.shape}")
print(f" - Kích thước tập Val   sau tiền xử lý: {df_val_proc.shape}")
print(f" - Kích thước tập Test  sau tiền xử lý: {df_test_proc.shape}")
print(f" - Số lượng ô NaN còn lại: Train={df_train_proc.isna().sum().sum()}, Val={df_val_proc.isna().sum().sum()}, Test={df_test_proc.isna().sum().sum()}")
print("-" * 85)
print("5 dòng đầu tiên của tập Train sau tiền xử lý:")
display(df_train_proc.head())

KÍCH THƯỚC CÁC TẬP DỮ LIỆU SAU PHÂN CHIA (70% Train - 15% Val - 15% Test):
 - Tập Huấn luyện (Train) : 35,000 mẫu (70.0%) | Tỷ lệ duyệt: 55.05%
 - Tập Kiểm định  (Val)   : 7,500 mẫu  (15.0%) | Tỷ lệ duyệt: 55.05%
 - Tập Kiểm tra   (Test)  : 7,500 mẫu  (15.0%) | Tỷ lệ duyệt: 55.04%

KẾT QUẢ TIỀN XỬ LÝ MA TRẬN ĐẶC TRƯNG:
 - Tổng số đặc trưng sau mã hóa: 24 đặc trưng
 - Kích thước tập Train sau tiền xử lý: (35000, 24)
 - Kích thước tập Val   sau tiền xử lý: (7500, 24)
 - Kích thước tập Test  sau tiền xử lý: (7500, 24)
 - Số lượng ô NaN còn lại: Train=0, Val=0, Test=0
-------------------------------------------------------------------------------------
5 dòng đầu tiên của tập Train sau tiền xử lý:


,num__age,num__years_employed,num__annual_income,num__credit_score,num__credit_history_years,num__savings_assets,num__current_debt,num__defaults_on_file,num__delinquencies_last_2yrs,num__derogatory_marks,...,num__payment_to_income_ratio,cat__occupation_status_Self-Employed,cat__occupation_status_Student,cat__product_type_Line of Credit,cat__product_type_Personal Loan,cat__loan_intent_Debt Consolidation,cat__loan_intent_Education,cat__loan_intent_Home Improvement,cat__loan_intent_Medical,cat__loan_intent_Personal
21145,-1.071920,-0.837328,-1.070557,-0.408317,-0.660966,-0.264351,-0.927615,-0.238853,-0.658469,-0.357831,...,-0.498894,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
42122,-1.521365,-0.982074,-0.273827,0.487732,-1.104751,-0.268433,-0.553127,-0.238853,-0.658469,-0.357831,...,-1.161196,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15264,0.276417,0.807520,-0.432780,-0.516461,0.989357,-0.108184,-0.729697,-0.238853,0.527290,-0.357831,...,-0.248120,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
12561,-0.262918,0.570662,-0.879058,-1.520654,-0.397470,0.082225,-0.834984,4.186684,2.898806,2.059946,...,-0.756099,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
42372,-0.892142,-0.560993,0.531653,-0.408317,-0.175577,0.388057,-0.311067,-0.238853,-0.658469,-0.357831,...,-0.820400,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
